In [1]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

In [2]:
env_name = "LunarLander-v3"
env = make_vec_env(env_name, n_envs=16)

print("Observation Space Shape : ", env.observation_space.shape)
print("Sample Observation : ", env.observation_space.sample())

Observation Space Shape :  (8,)
Sample Observation :  [-0.16295564 -1.1109498   8.363416   -2.6933947   2.912297    2.8697398
  0.3150559   0.95377827]


C:\Users\ashut\anaconda3\envs\PbRL_env\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [3]:
print("Action Space Shape : ", env.action_space.n)
print("Action Space Sample : ", env.action_space.sample())

Action Space Shape :  4
Action Space Sample :  0


In [4]:
model = PPO(
    policy = "MlpPolicy", 
    env = env, 
    n_steps = 1024, 
    batch_size = 64, 
    n_epochs = 4, 
    gamma = 0.999, 
    gae_lambda = 0.98, 
    ent_coef = 0.01, 
    verbose = 1, 
    device = "cpu"
)

Using cpu device


In [5]:
model.learn(total_timesteps= 1000000)

model_name = "ppo-LunarLander-v3"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 88.8     |
|    ep_rew_mean     | -190     |
| time/              |          |
|    fps             | 8509     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 111         |
|    ep_rew_mean          | -143        |
| time/                   |             |
|    fps                  | 3341        |
|    iterations           | 2           |
|    time_elapsed         | 9           |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.005470763 |
|    clip_fraction        | 0.0227      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | 0.00203     |
|    learning_rate        | 0.

In [6]:
from stable_baselines3.common.evaluation import evaluate_policy

In [7]:
#Evaluation : 

model = PPO.load("ppo-LunarLander-v3.zip", device="cpu")

eval_env = Monitor(gym.make("LunarLander-v3"))

print("Evaluating model quality")
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)

print(f"Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

Evaluating model quality
Mean Reward: 228.76 +/- 25.51


In [8]:
env = gym.make("LunarLander-v3", render_mode = "human")

print("Random Agent : Before Training")

obs, _ = env.reset()
for i in range (300):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

Random Agent : Before Training


In [ ]:
import pygame

env = gym.make("LunarLander-v3", render_mode = "human")

print("Trained Agent : After Training")

obs, _ = env.reset()

try:
    while True:
        
        for event in pygame.event.get():
            
            # If user presses a key
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    raise KeyboardInterrupt
                
        action, _ = model.predict(obs, deterministic=True)
        
        obs, reward, terminated, truncated, info = env.step(action)
        
        if terminated or truncated:
            print("Episode over! Resetting")
            obs, _ = env.reset()
            
except KeyboardInterrupt:
    print("\nClosing Environment")
    env.close()